# Lecture 3: Password Attack with Differential-Power-Analysis (Kocher et al. 1999)

In [1]:
%load_ext autoreload
%autoreload 2

import os
import random

import lascar
import numpy as np
import plotly.graph_objects as pgo
from cwtoolbox import CaptureDevice

In [2]:
capture_device = CaptureDevice.create("CWLITEXMEGA")
capture_device.compile(file=os.path.abspath("sbox_lookup.c"))
capture_device.flash()

c:\work\securecoding_ws2526\.venv\Lib\site-packages\chipwhisperer\capture\trace\TraceWhisperer.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources # type: ignore


XMEGA Programming flash...
XMEGA Reading flash...
Verified flash OK, 2553 bytes


In [3]:
data = capture_device.capture(
    number_of_traces=1000,
    input=lambda _: [random.randint(0, 255)] + 15 * [0],
)

100%|██████████| 1000/1000 [00:23<00:00, 42.10it/s]


In [5]:
def selection_function(value, guess):
    return lascar.tools.aes.sbox[value["input"][0] ^ guess] & 0x80 == 0


trace = lascar.TraceBatchContainer(data["trace"], data)
engine = lascar.DpaEngine(
    name="dpa",
    selection_function=selection_function,
    guess_range=range(256),
)

session = lascar.Session(
    trace,
    engine=engine,
    output_method=lascar.TableOutputMethod(engine),
)
session.run(batch_size="auto")

2026-01-09 09:47:48,115 - lascar.session - INFO - Session Session: 1000 traces, 3 engines, batch_size=1299514, leakage_shape=(1376,)
INFO:lascar.session:Session Session: 1000 traces, 3 engines, batch_size=1299514, leakage_shape=(1376,)
Session |  0%||0 trc/1000 | (3 engines, batch_size=1299514, leakage_shape=(1376,)) |ETA:  --:--:--
2026-01-09 09:47:51,217 - lascar.output.output_method - INFO - dpa
INFO:lascar.output.output_method:dpa
Session |100%||1000 trc/1000 | (3 engines, batch_size=1299514, leakage_shape=(1376,)) |ETA:  00:00:00
Session |100%||1000 trc/1000 | (3 engines, batch_size=1299514, leakage_shape=(1376,)) |Time:  0:00:03


                 dpa
    rank 1         1
                 0.0
    rank 2       133
                 0.0
    rank 3        62
                 0.0
    rank 4       112
                 0.0
    rank 5       104
                 0.0
    rank 6       118
                 0.0
    rank 7       147
                 0.0
    rank 8       197
                 0.0
    rank 9       240
                 0.0
   rank 10       228
                 0.0

